# Texas USPS Facility Mapping to SQL

This notebook converts `usps_texas_facilities_final.csv` into a MySQL-ready facility import file named `postal_tx_facilities.csv`. Each output row preserves the same unique USPS facility represented in the source CSV while mapping only the fields needed for the `facility` table.

## 1. Title and purpose

The target MySQL schema uses the `facility` and `facility_type` tables. This notebook:

- combines `PO Name` and `unit_names_seen` into a unique `facility_name`
- selects one primary `facility_type_name` from semicolon-separated source values
- maps that primary type to the existing `facility_type_id`
- maps address fields directly into import-ready columns
- leaves `territory_id` blank for now
- validates that the output is safe to review before import

## 2. Imports and file paths

In [14]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


def first_existing_path(*relative_candidates):
    cwd = Path.cwd().resolve()
    search_roots = [cwd, *cwd.parents]
    for root in search_roots:
        for candidate in relative_candidates:
            candidate_path = candidate if isinstance(candidate, Path) else Path(candidate)
            path = (root / candidate_path).resolve()
            if path.exists():
                return path
    raise FileNotFoundError(f"Could not find any of: {relative_candidates}")


SOURCE_PATH = first_existing_path(
    Path("Synthetic MySQL Tables") / "Facilities" / "usps_texas_facilities_final.csv",
    Path("usps_texas_facilities_final.csv"),
)
OUTPUT_PATH = SOURCE_PATH.with_name("postal_tx_facilities.csv")
AUDIT_PATH = SOURCE_PATH.with_name("postal_tx_facilities_audit_issues.csv")

FINAL_COLUMNS = [
    "facility_name",
    "facility_type_id",
    "facility_type_name",
    "county",
    "street_address",
    "city",
    "state_code",
    "zip_code",
    "territory_id",
]

TYPE_PRIORITY = [
    "Mail Processing",
    "Network Facilities",
    "Administrative Office",
    "Vehicle Maintenance",
    "Post Office",
]

FACILITY_TYPE_ID_MAP = {
    "Post Office": 1,
    "Vehicle Maintenance": 2,
    "Administrative Office": 3,
    "Network Facilities": 4,
    "Mail Processing": 5,
}

print(f"Source CSV: {SOURCE_PATH}")
print(f"Output CSV: {OUTPUT_PATH}")
print(f"Audit CSV:  {AUDIT_PATH}")

Source CSV: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Facilities\usps_texas_facilities_final.csv
Output CSV: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Facilities\postal_tx_facilities.csv
Audit CSV:  Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Facilities\postal_tx_facilities_audit_issues.csv


## 3. Load source CSV

In [15]:
source_df = pd.read_csv(
    SOURCE_PATH,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)
working_df = source_df.copy()
working_df["source_row_number"] = working_df.index + 2

print(f"Input CSV loaded successfully: {SOURCE_PATH.name}")
print(f"Loaded {len(source_df):,} rows.")

Input CSV loaded successfully: usps_texas_facilities_final.csv
Loaded 538 rows.


## 4. Inspect source shape and columns

In [16]:
print("Source shape:", source_df.shape)
print("\nSource columns:")
for column_name in source_df.columns:
    print(f"- {column_name}")

print("\nSource preview:")
print(source_df.head(5).to_string(index=False))

Source shape: (538, 21)

Source columns:
- District
- PO Name
- Property Address
- County
- City
- ST
- ZIP Code
- zip5
- zip_suffix
- Ownership
- facility_types
- facility_subtypes
- unit_names_seen
- Int Sq Ft
- Bldg Occu Date
- source_row_count
- source_row_ids
- original_zip_codes
- original_facility_types
- original_facility_subtypes
- review_reasons

Source preview:
District PO Name    Property Address  County    City ST   ZIP Code  zip5 zip_suffix Ownership      facility_types                  facility_subtypes    unit_names_seen Int Sq Ft Bldg Occu Date source_row_count source_row_ids original_zip_codes original_facility_types         original_facility_subtypes review_reasons
 TEXAS 3 ABILENE 2501 BUFFALO GAP RD  TAYLOR ABILENE TX 79605-6185 79605       6185     Owned         Post Office                            Station SOUTHERN HILLS STA    17,130      12/1/1984                1              3         79605-6185             Post Office                            Station     

## 5. Define helper functions

In [17]:
def normalize_text(value):
    if value is None or pd.isna(value):
        return pd.NA
    text = str(value).strip()
    return text if text else pd.NA


def split_facility_types(value):
    text = normalize_text(value)
    if pd.isna(text):
        return []
    return [part.strip() for part in str(text).split(";") if part.strip()]


def create_facility_name(row):
    po_name = normalize_text(row["PO Name"])
    unit_name = normalize_text(row["unit_names_seen"])
    if pd.isna(po_name) or pd.isna(unit_name):
        return pd.NA
    return f"{po_name} - {unit_name}"


def choose_primary_facility_type(value):
    parsed_types = split_facility_types(value)
    for facility_type_name in TYPE_PRIORITY:
        if facility_type_name in parsed_types:
            return facility_type_name
    return pd.NA


def format_zip5(value):
    text = normalize_text(value)
    if pd.isna(text):
        return pd.NA
    return str(text).zfill(5)


def print_rows(label, frame, columns=None, max_rows=10):
    print(label)
    if frame.empty:
        print("None")
        return

    preview = frame if columns is None else frame.loc[:, columns]
    print(preview.head(max_rows).to_string(index=False))
    if len(preview) > max_rows:
        print(f"... showing first {max_rows} of {len(preview)} rows")

## 6. Create `facility_name`

In [18]:
working_df["facility_name"] = working_df.apply(create_facility_name, axis=1)

missing_name_source_mask = (
    working_df["PO Name"].map(normalize_text).isna()
    | working_df["unit_names_seen"].map(normalize_text).isna()
)
missing_name_source_rows = working_df.loc[missing_name_source_mask]

print(f"Created facility_name values for {(~working_df['facility_name'].isna()).sum():,} rows.")
print_rows(
    "Rows missing PO Name or unit_names_seen:",
    missing_name_source_rows,
    columns=["source_row_number", "PO Name", "unit_names_seen", "Property Address", "City", "ST", "zip5"],
)

Created facility_name values for 538 rows.
Rows missing PO Name or unit_names_seen:
None


## 7. Select primary `facility_type_name`

In [19]:
working_df["facility_type_name"] = working_df["facility_types"].map(choose_primary_facility_type)

unmapped_type_rows = working_df.loc[working_df["facility_type_name"].isna()]

print("Distinct mapped facility_type_name values:")
print(sorted(working_df["facility_type_name"].dropna().unique().tolist()))
print_rows(
    "Rows with no mapped primary facility type:",
    unmapped_type_rows,
    columns=["source_row_number", "PO Name", "unit_names_seen", "facility_types"],
)

Distinct mapped facility_type_name values:
['Administrative Office', 'Mail Processing', 'Network Facilities', 'Post Office', 'Vehicle Maintenance']
Rows with no mapped primary facility type:
None


## 8. Map `facility_type_id`

In [20]:
working_df["facility_type_id"] = working_df["facility_type_name"].map(FACILITY_TYPE_ID_MAP)

print("facility_type_name to facility_type_id mapping used in this export:")
print(
    working_df[["facility_type_name", "facility_type_id"]]
    .drop_duplicates()
    .sort_values("facility_type_id")
    .to_string(index=False)
)

facility_type_name to facility_type_id mapping used in this export:
   facility_type_name  facility_type_id
          Post Office                 1
  Vehicle Maintenance                 2
Administrative Office                 3
   Network Facilities                 4
      Mail Processing                 5


## 9. Map address/location fields

In [21]:
working_df["county"] = working_df["County"].map(normalize_text)
working_df["street_address"] = working_df["Property Address"].map(normalize_text)
working_df["city"] = working_df["City"].map(normalize_text)
working_df["state_code"] = working_df["ST"].map(normalize_text)
working_df["zip_code"] = working_df["zip5"].map(format_zip5)

print("Mapped address/location fields preview:")
print(
    working_df[["county", "street_address", "city", "state_code", "zip_code"]]
    .head(5)
    .to_string(index=False)
)

Mapped address/location fields preview:
 county      street_address    city state_code zip_code
 TAYLOR 2501 BUFFALO GAP RD ABILENE         TX    79605
 TAYLOR         341 PINE ST ABILENE         TX    79601
 TAYLOR        810 N 4TH ST ABILENE         TX    79601
 DALLAS   4900 AIRPORT PKWY ADDISON         TX    75001
HIDALGO   423 LOS ALAMOS DR   ALAMO         TX    78516


## 10. Add blank `territory_id`

In [22]:
working_df["territory_id"] = pd.Series(pd.NA, index=working_df.index, dtype="object")
print("territory_id was set to blank/NULL for every row.")

territory_id was set to blank/NULL for every row.


## 11. Build final dataframe

In [23]:
final_df = working_df.loc[:, FINAL_COLUMNS].copy()

print("Final dataframe shape:", final_df.shape)
print("\nFinal dataframe preview:")
print(final_df.head(5).to_string(index=False))

Final dataframe shape: (538, 9)

Final dataframe preview:
               facility_name  facility_type_id  facility_type_name  county      street_address    city state_code zip_code territory_id
ABILENE - SOUTHERN HILLS STA                 1         Post Office  TAYLOR 2501 BUFFALO GAP RD ABILENE         TX    79605         <NA>
       ABILENE - MAIN OFFICE                 1         Post Office  TAYLOR         341 PINE ST ABILENE         TX    79601         <NA>
     ABILENE - AUXILIARY VMF                 2 Vehicle Maintenance  TAYLOR        810 N 4TH ST ABILENE         TX    79601         <NA>
       ADDISON - MAIN OFFICE                 1         Post Office  DALLAS   4900 AIRPORT PKWY ADDISON         TX    75001         <NA>
         ALAMO - MAIN OFFICE                 1         Post Office HIDALGO   423 LOS ALAMOS DR   ALAMO         TX    78516         <NA>


## 12. Run validation checks

In [24]:
audit_frames = []
hard_failures = []
valid_type_ids = set(FACILITY_TYPE_ID_MAP.values())


def add_audit_issue(issue_name, frame):
    if frame.empty:
        return
    audit_frame = frame.copy()
    audit_frame.insert(0, "audit_issue", issue_name)
    preferred_columns = [
        "audit_issue",
        "source_row_number",
        "PO Name",
        "unit_names_seen",
        "facility_types",
        "facility_name",
        "facility_type_name",
        "facility_type_id",
        "Property Address",
        "City",
        "ST",
        "zip5",
        "street_address",
        "city",
        "state_code",
        "zip_code",
        "territory_id",
    ]
    available_columns = [column for column in preferred_columns if column in audit_frame.columns]
    audit_frames.append(audit_frame.loc[:, available_columns])


print("Validation 1. Confirm the input CSV loads successfully.")
if len(source_df) > 0:
    print(f"PASS: Loaded {len(source_df):,} source rows.")
else:
    message = "Input CSV is empty."
    print(f"FAIL: {message}")
    hard_failures.append(message)

print("\nValidation 2. Confirm the source row count and output row count are equal.")
print(f"Source rows: {len(source_df):,}")
print(f"Output rows: {len(final_df):,}")
if len(source_df) == len(final_df):
    print("PASS: Source and output row counts match.")
else:
    message = f"Row count mismatch: source={len(source_df)}, output={len(final_df)}"
    print(f"FAIL: {message}")
    hard_failures.append(message)

print("\nValidation 3. Confirm no rows were dropped.")
if len(source_df) == len(final_df):
    print("PASS: No rows were dropped.")
else:
    message = "Rows were dropped during transformation."
    print(f"FAIL: {message}")
    hard_failures.append(message)

print("\nValidation 4. Confirm final output has exactly the requested columns in the requested order.")
print("Expected columns:", FINAL_COLUMNS)
print("Actual columns:  ", list(final_df.columns))
if list(final_df.columns) == FINAL_COLUMNS:
    print("PASS: Final columns match the requested schema and order.")
else:
    message = "Final output columns do not match the requested schema or order."
    print(f"FAIL: {message}")
    hard_failures.append(message)

facility_name_blank_mask = final_df["facility_name"].isna() | final_df["facility_name"].astype("string").str.strip().eq("")
print("\nValidation 5. Confirm facility_name is not null or blank.")
print_rows(
    "Affected rows:",
    working_df.loc[facility_name_blank_mask],
    columns=["source_row_number", "PO Name", "unit_names_seen", "facility_name", "Property Address", "City", "ST", "zip5"],
)
if facility_name_blank_mask.any():
    message = f"Found {facility_name_blank_mask.sum()} rows with blank facility_name."
    print(f"FAIL: {message}")
    hard_failures.append(message)
    add_audit_issue("blank_facility_name", working_df.loc[facility_name_blank_mask])
else:
    print("PASS: facility_name is populated for every row.")

facility_type_name_null_mask = final_df["facility_type_name"].isna()
print("\nValidation 6. Confirm facility_type_name is not null.")
print_rows(
    "Affected rows:",
    working_df.loc[facility_type_name_null_mask],
    columns=["source_row_number", "PO Name", "unit_names_seen", "facility_types", "facility_type_name"],
)
if facility_type_name_null_mask.any():
    message = f"Found {facility_type_name_null_mask.sum()} rows with null facility_type_name."
    print(f"FAIL: {message}")
    hard_failures.append(message)
    add_audit_issue("missing_facility_type_name", working_df.loc[facility_type_name_null_mask])
else:
    print("PASS: facility_type_name is populated for every row.")

facility_type_id_null_mask = final_df["facility_type_id"].isna()
print("\nValidation 7. Confirm facility_type_id is not null.")
print_rows(
    "Affected rows:",
    working_df.loc[facility_type_id_null_mask],
    columns=["source_row_number", "PO Name", "unit_names_seen", "facility_types", "facility_type_name", "facility_type_id"],
)
if facility_type_id_null_mask.any():
    message = f"Found {facility_type_id_null_mask.sum()} rows with null facility_type_id."
    print(f"FAIL: {message}")
    hard_failures.append(message)
    add_audit_issue("missing_facility_type_id", working_df.loc[facility_type_id_null_mask])
else:
    print("PASS: facility_type_id is populated for every row.")

required_field_masks = {
    "street_address": final_df["street_address"].isna() | final_df["street_address"].astype("string").str.strip().eq(""),
    "city": final_df["city"].isna() | final_df["city"].astype("string").str.strip().eq(""),
    "state_code": final_df["state_code"].isna() | final_df["state_code"].astype("string").str.strip().eq(""),
    "zip_code": final_df["zip_code"].isna() | final_df["zip_code"].astype("string").str.strip().eq(""),
}
required_field_missing_mask = pd.DataFrame(required_field_masks).any(axis=1)
print("\nValidation 8. Confirm street_address, city, state_code, and zip_code are not null.")
print_rows(
    "Affected rows:",
    working_df.loc[required_field_missing_mask],
    columns=["source_row_number", "facility_name", "street_address", "city", "state_code", "zip_code"],
)
if required_field_missing_mask.any():
    message = f"Found {required_field_missing_mask.sum()} rows with missing required location fields."
    print(f"FAIL: {message}")
    hard_failures.append(message)
    add_audit_issue("missing_required_location_fields", working_df.loc[required_field_missing_mask])
else:
    print("PASS: Required location fields are populated for every row.")

state_invalid_mask = final_df["state_code"].astype("string").str.strip().ne("TX")
print("\nValidation 9. Confirm every state_code is TX.")
print_rows(
    "Affected rows:",
    working_df.loc[state_invalid_mask],
    columns=["source_row_number", "facility_name", "state_code"],
)
if state_invalid_mask.any():
    message = f"Found {state_invalid_mask.sum()} rows where state_code is not TX."
    print(f"FAIL: {message}")
    hard_failures.append(message)
    add_audit_issue("invalid_state_code", working_df.loc[state_invalid_mask])
else:
    print("PASS: Every state_code is TX.")

zip_invalid_mask = ~final_df["zip_code"].astype("string").str.fullmatch(r"\d{5}", na=False)
print("\nValidation 10. Confirm every zip_code is exactly 5 characters.")
print_rows(
    "Affected rows:",
    working_df.loc[zip_invalid_mask],
    columns=["source_row_number", "facility_name", "zip_code"],
)
if zip_invalid_mask.any():
    message = f"Found {zip_invalid_mask.sum()} rows with invalid zip_code values."
    print(f"FAIL: {message}")
    hard_failures.append(message)
    add_audit_issue("invalid_zip_code", working_df.loc[zip_invalid_mask])
else:
    print("PASS: Every zip_code is exactly 5 digits.")

facility_type_id_invalid_mask = ~final_df["facility_type_id"].isin(valid_type_ids)
print("\nValidation 11. Confirm facility_type_id values are only 1, 2, 3, 4, or 5.")
print_rows(
    "Affected rows:",
    working_df.loc[facility_type_id_invalid_mask],
    columns=["source_row_number", "facility_name", "facility_type_name", "facility_type_id"],
)
if facility_type_id_invalid_mask.any():
    message = f"Found {facility_type_id_invalid_mask.sum()} rows with invalid facility_type_id values."
    print(f"FAIL: {message}")
    hard_failures.append(message)
    add_audit_issue("invalid_facility_type_id", working_df.loc[facility_type_id_invalid_mask])
else:
    print("PASS: facility_type_id values are restricted to 1, 2, 3, 4, or 5.")

duplicate_facility_name_mask = final_df.duplicated(subset=["facility_name"], keep=False)
print("\nValidation 12. Check for duplicate facility_name values.")
print_rows(
    "Affected rows:",
    working_df.loc[duplicate_facility_name_mask].sort_values(["facility_name", "street_address", "zip_code"]),
    columns=["source_row_number", "facility_name", "facility_type_name", "street_address", "city", "state_code", "zip_code"],
)
if duplicate_facility_name_mask.any():
    print(f"WARNING: Found {duplicate_facility_name_mask.sum()} rows with duplicate facility_name values.")
    add_audit_issue("duplicate_facility_name", working_df.loc[duplicate_facility_name_mask])
else:
    print("PASS: No duplicate facility_name values found.")

duplicate_location_mask = final_df.duplicated(
    subset=["street_address", "city", "state_code", "zip_code"],
    keep=False,
)
print("\nValidation 13. Check for duplicate physical locations.")
print_rows(
    "Affected rows:",
    working_df.loc[duplicate_location_mask].sort_values(["street_address", "city", "zip_code", "facility_name"]),
    columns=["source_row_number", "facility_name", "street_address", "city", "state_code", "zip_code"],
)
if duplicate_location_mask.any():
    print(f"WARNING: Found {duplicate_location_mask.sum()} rows with duplicate physical locations.")
    add_audit_issue("duplicate_physical_location", working_df.loc[duplicate_location_mask])
else:
    print("PASS: No duplicate physical locations found.")

facility_name_length_mask = final_df["facility_name"].astype("string").str.len().gt(100)
print("\nValidation 14. Check that facility_name length is less than or equal to 100 characters.")
print_rows(
    "Affected rows:",
    working_df.loc[facility_name_length_mask].sort_values("facility_name"),
    columns=["source_row_number", "facility_name"],
)
if facility_name_length_mask.any():
    print(f"WARNING: Found {facility_name_length_mask.sum()} rows where facility_name exceeds 100 characters.")
    add_audit_issue("facility_name_exceeds_100_chars", working_df.loc[facility_name_length_mask])
else:
    print("PASS: All facility_name values are 100 characters or fewer.")

street_address_length_mask = final_df["street_address"].astype("string").str.len().gt(120)
print("\nValidation 15. Check that street_address length is less than or equal to 120 characters.")
print_rows(
    "Affected rows:",
    working_df.loc[street_address_length_mask].sort_values("street_address"),
    columns=["source_row_number", "facility_name", "street_address"],
)
if street_address_length_mask.any():
    print(f"WARNING: Found {street_address_length_mask.sum()} rows where street_address exceeds 120 characters.")
    add_audit_issue("street_address_exceeds_120_chars", working_df.loc[street_address_length_mask])
else:
    print("PASS: All street_address values are 120 characters or fewer.")

city_length_mask = final_df["city"].astype("string").str.len().gt(60)
print("\nValidation 16. Check that city length is less than or equal to 60 characters.")
print_rows(
    "Affected rows:",
    working_df.loc[city_length_mask].sort_values("city"),
    columns=["source_row_number", "facility_name", "city"],
)
if city_length_mask.any():
    print(f"WARNING: Found {city_length_mask.sum()} rows where city exceeds 60 characters.")
    add_audit_issue("city_exceeds_60_chars", working_df.loc[city_length_mask])
else:
    print("PASS: All city values are 60 characters or fewer.")

if missing_name_source_mask.any():
    add_audit_issue("missing_name_source_fields", working_df.loc[missing_name_source_mask])
if unmapped_type_rows.shape[0] > 0:
    add_audit_issue("unmapped_primary_facility_type", unmapped_type_rows)

if audit_frames:
    audit_df = pd.concat(audit_frames, ignore_index=True).drop_duplicates()
    print(f"\nAudit issues collected: {len(audit_df):,} rows.")
else:
    audit_df = pd.DataFrame()
    print("\nNo audit issues collected.")

if hard_failures:
    raise AssertionError("Validation failed:\n- " + "\n- ".join(hard_failures))

print("\nAll assertion-based validation checks passed.")

Validation 1. Confirm the input CSV loads successfully.
PASS: Loaded 538 source rows.

Validation 2. Confirm the source row count and output row count are equal.
Source rows: 538
Output rows: 538
PASS: Source and output row counts match.

Validation 3. Confirm no rows were dropped.
PASS: No rows were dropped.

Validation 4. Confirm final output has exactly the requested columns in the requested order.
Expected columns: ['facility_name', 'facility_type_id', 'facility_type_name', 'county', 'street_address', 'city', 'state_code', 'zip_code', 'territory_id']
Actual columns:   ['facility_name', 'facility_type_id', 'facility_type_name', 'county', 'street_address', 'city', 'state_code', 'zip_code', 'territory_id']
PASS: Final columns match the requested schema and order.

Validation 5. Confirm facility_name is not null or blank.
Affected rows:
None
PASS: facility_name is populated for every row.

Validation 6. Confirm facility_type_name is not null.
Affected rows:
None
PASS: facility_type_nam

## 13. Export `postal_tx_facilities.csv`

In [25]:
final_df.to_csv(OUTPUT_PATH, index=False, na_rep="")
print(f"Exported {len(final_df):,} rows to {OUTPUT_PATH}")

Exported 538 rows to Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Facilities\postal_tx_facilities.csv


## 14. Optional issue report / audit output

In [26]:
if not audit_df.empty:
    audit_df.to_csv(AUDIT_PATH, index=False, na_rep="")
    print(f"Audit file created: {AUDIT_PATH}")
    print(audit_df.head(20).to_string(index=False))
else:
    print("No audit issues detected. Audit file was not created.")

No audit issues detected. Audit file was not created.
